# Get Energy spectra by trigger, scaled and unscaled

In [1]:
import ROOT, glob
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

plt.rcParams['figure.dpi'] = 400
plt.rcParams['savefig.dpi'] = 400

In [2]:
dirDir= "/home/users/l/lebrun/"
dataDir = "/dpnc/beegfs/dampe/tryf/MC_Skimmed/MC_gamma/"
figPath_ = dirDir + "figures/"
resultPath = dirDir + "results/"
dataPath = dirDir + "data/"
# filename1 = dataDir + "DmlNtup_gamma_part*_TQ_FullSel.root"
# filename2 = dataDir + "DmlNtup_gamma_100GeV_1TeV_part*_TQ_FullSel.root"
# filename3 = dataDir + "DmlNtup_gamma_1TeV_10TeV_part*_MPhotonSelProj_TQ_FullSel.root"
filename1 = dataDir + "DmlNtup_gamma_1GeV_1TeV_comb_kim_TQ_FUllSel.root"
filename2 = dataDir + "DmlNtup_gamma_combined_kim_TQ_FullSel.root"
filenames_LET = [filename1]
filenames_HET = [filename2]

# filenames_LET = [filename1, filename2]
# filenames_HET = [filename1, filename2, filename3]

# Find all matching ROOT files : from 1 GeV to 100 GeV
files = list()
for filename in filenames_HET:
    files += sorted(glob.glob(filename))
exponent = 0

print(f"Number of files found : {len(files)}")
print(files)

file = dataPath + f"MC_exp{exponent:.1f}.root"

trees = "DmlNtup"
tree = "MC"

# Pass the list of files to RDataFrame
rdf = ROOT.RDataFrame(tree, file)

# Apply cut on FoV
#rdf = rdf.Filter("theta_True < 60")


Number of files found : 1
['/dpnc/beegfs/dampe/tryf/MC_Skimmed/MC_gamma/DmlNtup_gamma_combined_kim_TQ_FullSel.root']


## TODO : remove when batch job over

In [3]:
COUNTER = 0

def create_log_bins(min_val, max_val, n_bins):
    """Create logarithmically spaced bin edges"""
    return np.logspace(np.log10(min_val), np.log10(max_val), n_bins + 1)

def declare_fit_function():
    cpp_code = f"""
        #include <cmath>        
        double powerLaw(double *x, double a, double b){{
            return b*pow(x[0], a);
        }}
        double fitFunction(double *x, double *par){{
            return powerLaw(x, par[0], par[1]);
        }}
        """
        
    ROOT.gInterpreter.Declare(cpp_code)

In [4]:
def rescale(rdf, Emin=1, Emax=1e4, exponent=-2.5, reference_energy=1, n_bins = 50, energy_column:str ="tt_ekin", figPath=None):
    """ Rescale the energy of the events in the RDataFrame
    
    Args:
        rdf (ROOT.RDataFrame): The input RDataFrame.
        Emin (float): Minimum energy in GeV.
        Emax (float): Maximum energy in GeV.
        exponent (float): Exponent for the energy rescaling.
        reference_energy (float): Reference energy for filtering in GeV.
        n_bins (int): Number of bins for the logarithmic binning.
        energy_column (str): Name of the column containing the energy values in MeV.
        
    Returns:
        ROOT.RDataFrame: The RDataFrame with rescaled energy."""
    
    # Rescale the energy
    rdf = rdf.Define("E_true", energy_column + " / 1000.0")


    def create_histogram_based_weights_root(rdf, Emin=1, Emax=1e4, exponent=-2.5, reference_energy=1, n_bins = 50, energy_column:str ="E_true"):
        """
        Create histogram-based power law weights in ROOT to smooth out step-like behavior
        This mimics the pandas approach that worked well
        """
        global COUNTER

        COUNTER += 1

        # Create logarithmic bins
        log_bins = create_log_bins(Emin, Emax, n_bins)
        log_bins_root = ROOT.std.vector('double')(log_bins)
        
        # Get current histogram
        h_current = rdf.Histo1D(ROOT.RDF.TH1DModel("h_current", "Current Distribution", 
                                                n_bins, log_bins_root.data()), energy_column)
        
        # Force evaluation to get the histogram
        h_current_ptr = h_current.GetPtr()
        
        # Step 2: Calculate target distribution (power law)
        bin_centers = []
        current_counts = []
        target_counts = []
        
        for i in range(1, n_bins + 1):  # ROOT bins start at 1
            bin_center = h_current_ptr.GetBinCenter(i)
            current_count = h_current_ptr.GetBinContent(i)
            
            bin_centers.append(bin_center)
            current_counts.append(current_count)
            
            # Target count following power law
            target_count = (bin_center/reference_energy) ** exponent
            target_counts.append(target_count)
        
        # Normalize target to same total as current
        total_current = sum(current_counts)
        total_target = sum(target_counts)
        
        if total_target > 0:
            normalization = total_current / total_target
            target_counts = [t * normalization for t in target_counts]
        
        # Step 3: Calculate bin weights
        bin_weights = []
        for current, target in zip(current_counts, target_counts):
            if current > 0:
                weight = target / current
            else:
                weight = 1.0  # Default weight for empty bins
            bin_weights.append(weight)
        
        # Step 4: Create C++ function that maps energy to appropriate bin weight
        # First, create arrays for the C++ function
        bin_edges_str = "{" + ", ".join(map(str, log_bins)) + "}"
        bin_weights_str = "{" + ", ".join(map(str, bin_weights)) + "}"
        
        cpp_code = f"""
        #include <algorithm>
        #include <vector>
        
        double GetHistogramBasedWeight{COUNTER}(double energy) {{
            static std::vector<double> bin_edges = {bin_edges_str};
            static std::vector<double> bin_weights = {bin_weights_str};
            
            // Find bin index
            auto it = std::upper_bound(bin_edges.begin(), bin_edges.end(), energy);
            int bin_idx = std::distance(bin_edges.begin(), it) - 1;
            
            // Clamp to valid range
            bin_idx = std::max(0, std::min(bin_idx, (int)bin_weights.size() - 1));
            
            return bin_weights[bin_idx];
        }}
        """
        
        ROOT.gInterpreter.Declare(cpp_code)
        
        return log_bins_root

    def create_smooth_power_law_rdf(rdf, Emin=1, Emax=1e4, exponent=-2.5, reference_energy=1, n_bins = 50, energy_column:str ="E_true"):
        """
        Create RDF with histogram-based power law weights that smooth out steps
        """
        
        # Create the histogram-based weighting system
        log_bins_root = create_histogram_based_weights_root(rdf, Emin, Emax, exponent, reference_energy, n_bins, energy_column)
        
        # Add the sophisticated weight column
        rdf_weighted = rdf.Define("weight", f"GetHistogramBasedWeight{COUNTER}({energy_column})")
        
        return rdf_weighted, log_bins_root
    
    rdf_weighted, log_bins_root = create_smooth_power_law_rdf(rdf, Emin, Emax, exponent, reference_energy, n_bins, "E_true")

    if figPath:
        
        h_weighted = rdf_weighted.Histo1D(ROOT.RDF.TH1DModel("h_weighted", "Power Law Weighted;Energy [GeV];Weighted Counts", 
                                                                n_bins, log_bins_root.data()), "E_true", "weight")
        ROOT.gStyle.SetOptStat(0)
        # Plot weighted
        c2 = ROOT.TCanvas()
        ROOT.gPad.SetLogx()
        ROOT.gPad.SetLogy() 
        ROOT.gPad.SetGrid()

        h_weight = h_weighted.GetPtr()
        h_weight.SetLineColor(ROOT.kRed)
        h_weight.SetLineWidth(2)
        h_weight.GetXaxis().SetLabelOffset(0.005)
        h_weight.GetXaxis().SetTitleOffset(1.3)
        # Draw histogram
        h_weight.Draw()

        # perform the fit
        fitFuncE = ROOT.TF1(f"fitFunction", ROOT.fitFunction, xmin=Emin, xmax=Emax, npar=2)
        fitFuncE.SetParameter(0, -2.5) # a
        fitFuncE.SetParameter(1, 4)    # b 
        fitFuncE.SetLineColor(ROOT.kBlack)
        fitFuncE.SetLineWidth(2)

        # Fit with "R+" option to draw the fit function
        res = h_weight.Fit(fitFuncE, "R")

        # Add legend
        alpha = fitFuncE.GetParameter(0)
        beta = np.log10(fitFuncE.GetParameter(1))
        leg = ROOT.TLegend(0.55, 0.7, 0.82, 0.8)
        leg.AddEntry(fitFuncE, f"log_{{10}}(y) \propto {alpha:.2f} log_{{10}}(x)", "l")
        leg.SetTextSize(0.03)
        leg.Draw("same")

        # Final update before saving
        c2.Update()

        c2.SaveAs(figPath+f"Scaled_Spectrum_exp{exponent}.pdf")

    return rdf_weighted

In [5]:
#rdf = rescale(rdf, exponent=exponent, energy_column="tt_ekin")

# Print min and max E_true
print("Min E_true:", rdf.Min("E_true").GetValue())
print("Max E_true:", rdf.Max("E_true").GetValue())


Min E_true: 1.0000006282052336
Max E_true: 9999.96505258949


## Plot Unscaled

In [6]:
# Plot Energy histogram of df which belong to neither HET nor HET
def create_log_bins(min_val, max_val, n_bins):
    """Create logarithmically spaced bin edges"""
    return np.logspace(np.log10(min_val), np.log10(max_val), n_bins + 1)


rdf_none = rdf.Filter("tt_isLET == 0").Filter("tt_isHET == 0")
rdf_LET = rdf.Filter("tt_isLET == 1")
rdf_LETnonHET = rdf.Filter("tt_isLET == 1").Filter("tt_isHET == 0")
rdf_HETnonLET = rdf.Filter("tt_isLET == 0").Filter("tt_isHET == 1")
rdf_HET = rdf.Filter("tt_isHET == 1")


### Plot 1-100 GeV (Unscaled)

In [7]:

n_bins = 30
min_val = 1
max_val = 100

log_bins = create_log_bins(min_val, max_val, n_bins)
log_bins_root = ROOT.std.vector('double')(log_bins)

h_all = rdf.Histo1D(("hAll", "Spectra by trigger type (1-100 GeV) (Unscaled); Energy [GeV]; Counts", n_bins, log_bins_root.data()), 'E_true')
h_none = rdf_none.Histo1D(("hNone", "; Energy [GeV]; Counts",n_bins, log_bins_root.data()), 'E_true')
h_LET = rdf_LET.Histo1D(("hLET", "; Energy [GeV]; Counts", n_bins, log_bins_root.data()), 'E_true')
h_LETnonHET = rdf_LETnonHET.Histo1D(("hLETnonHET", "; Energy [GeV]; Counts", n_bins, log_bins_root.data()), 'E_true')
h_HETnonLET = rdf_HETnonLET.Histo1D(("hHETnonLET", "; Energy [GeV]; Counts", n_bins, log_bins_root.data()), 'E_true')
h_HET = rdf_HET.Histo1D(("hHET", "; Energy [GeV]; Counts", n_bins, log_bins_root.data()), 'E_true')

In [8]:

# Style
h_none.SetLineColor(ROOT.kBlack)
h_LET.SetLineColor(ROOT.kBlue)
h_LETnonHET.SetLineColor(ROOT.kCyan+1)
h_HETnonLET.SetLineColor(ROOT.kRed)
h_HET.SetLineColor(ROOT.kMagenta)
h_all.SetLineColor(ROOT.kGreen+2)

h_none.SetMarkerStyle(1)
h_LET.SetMarkerStyle(1)
h_LETnonHET.SetMarkerStyle(1)
h_HETnonLET.SetMarkerStyle(1)
h_HET.SetMarkerStyle(1)
h_all.SetMarkerStyle(1)


h_none.GetXaxis().SetLabelOffset(0.005)
h_none.GetXaxis().SetTitleOffset(1.3)
h_LET.GetXaxis().SetLabelOffset(0.005)
h_LET.GetXaxis().SetTitleOffset(1.3)
h_LETnonHET.GetXaxis().SetLabelOffset(0.005)
h_LETnonHET.GetXaxis().SetTitleOffset(1.3)
h_HETnonLET.GetXaxis().SetLabelOffset(0.005)
h_HETnonLET.GetXaxis().SetTitleOffset(1.3)
h_HET.GetXaxis().SetLabelOffset(0.005)
h_HET.GetXaxis().SetTitleOffset(1.3)
h_all.GetXaxis().SetLabelOffset(0.005)
h_all.GetXaxis().SetTitleOffset(1.3)

# Draw
ROOT.gStyle.SetOptStat(0)
c = ROOT.TCanvas("1to100GeV", "1 to 100 GeV Spectra", 800, 600)

h_none.SetStats(0)

h_none.Draw("HIST")
h_LET.Draw("HIST SAME")
h_LETnonHET.Draw("HIST SAME")
h_HETnonLET.Draw("HIST SAME")
h_HET.Draw("HIST SAME")
h_all.Draw("HIST SAME")


legend = ROOT.TLegend(0.6, 0.6, 0.89, 0.89)  # x1, y1, x2, y2

legend.AddEntry(h_none.GetPtr(), "None", "l")
legend.AddEntry(h_LET.GetPtr(), "LET", "l")
legend.AddEntry(h_LETnonHET.GetPtr(), "LET only", "l")
legend.AddEntry(h_HETnonLET.GetPtr(), "HET only", "l")
legend.AddEntry(h_HET.GetPtr(), "HET", "l")
legend.AddEntry(h_all.GetPtr(), "All", "l")

legend.SetBorderSize(0)
legend.SetFillStyle(0)
legend.Draw("SAME")

c.SetLogy()
c.SetLogx()
c.SetTitle('Spectra by trigger type (1-100 GeV) (Unscaled)')
c.Update()
c.SaveAs(figPath_ + "scaling/" + "Spectrum_triggers_1_100.pdf")
c.Draw()

Info in <TCanvas::Print>: pdf file /home/users/l/lebrun/figures/scaling/Spectrum_triggers_1_100.pdf has been created


### Plot 1 GeV - 1 TeV (Unscaled)

In [9]:

n_bins = 60
min_val = 1
max_val = 10000

log_bins = create_log_bins(min_val, max_val, n_bins)
log_bins_root = ROOT.std.vector('double')(log_bins)

h_none2 = rdf_none.Histo1D(("hNone", "Spectra by trigger type (1 GeV-1 TeV) (Unscaled); Energy [GeV]; Counts",n_bins, log_bins_root.data()), 'E_true')
h_LET2 = rdf_LET.Histo1D(("hLET", "; Energy [GeV]; Counts", n_bins, log_bins_root.data()), 'E_true')
h_LETnonHET2 = rdf_LETnonHET.Histo1D(("hLETnonHET", "; Energy [GeV]; Counts", n_bins, log_bins_root.data()), 'E_true')
h_HETnonLET2 = rdf_HETnonLET.Histo1D(("hHETnonLET", "; Energy [GeV]; Counts", n_bins, log_bins_root.data()), 'E_true')
h_HET2 = rdf_HET.Histo1D(("hHET", "; Energy [GeV]; Counts", n_bins, log_bins_root.data()), 'E_true')
# Create a histogram for all events
h_all2 = rdf.Histo1D(("hAll", "; Energy [GeV]; Counts", n_bins, log_bins_root.data()), 'E_true')


# Style
h_none2.SetLineColor(ROOT.kBlack)
h_LET2.SetLineColor(ROOT.kBlue)
h_LETnonHET2.SetLineColor(ROOT.kCyan+1)
h_HETnonLET2.SetLineColor(ROOT.kRed)
h_HET2.SetLineColor(ROOT.kMagenta)
h_all2.SetLineColor(ROOT.kGreen+2)

h_none2.SetMarkerStyle(1)
h_LET2.SetMarkerStyle(1)
h_LETnonHET2.SetMarkerStyle(1)
h_HETnonLET2.SetMarkerStyle(1)
h_HET2.SetMarkerStyle(1)
h_all2.SetMarkerStyle(1)

h_none2.GetXaxis().SetLabelOffset(0.005)
h_none2.GetXaxis().SetTitleOffset(1.3)
h_LET2.GetXaxis().SetLabelOffset(0.005)
h_LET2.GetXaxis().SetTitleOffset(1.3)
h_LETnonHET2.GetXaxis().SetLabelOffset(0.005)
h_LETnonHET2.GetXaxis().SetTitleOffset(1.3)
h_HETnonLET2.GetXaxis().SetLabelOffset(0.005)
h_HETnonLET2.GetXaxis().SetTitleOffset(1.3)
h_HET2.GetXaxis().SetLabelOffset(0.005)
h_HET2.GetXaxis().SetTitleOffset(1.3)
h_all2.GetXaxis().SetLabelOffset(0.005)
h_all2.GetXaxis().SetTitleOffset(1.3)

# Draw
ROOT.gStyle.SetOptStat(0)
c2 = ROOT.TCanvas("1GeVto10TeV", "1 GeV to 10 TeV Spectra", 800, 600)

h_none2.SetStats(0)

h_none2.Draw("HIST")
h_LET2.Draw("HIST SAME")
h_LETnonHET2.Draw("HIST SAME")
h_HETnonLET2.Draw("HIST SAME")
h_HET2.Draw("HIST SAME")
h_all2.Draw("HIST SAME")


legend2 = ROOT.TLegend(0.71, 0.6, 0.94, 0.9)  # x1, y1, x2, y2

legend2.AddEntry(h_none.GetPtr(), "None", "l")
legend2.AddEntry(h_LET.GetPtr(), "LET", "l")
legend2.AddEntry(h_LETnonHET.GetPtr(), "LET only", "l")
legend2.AddEntry(h_HETnonLET.GetPtr(), "HET only", "l")
legend2.AddEntry(h_HET.GetPtr(), "HET", "l")
legend2.AddEntry(h_all.GetPtr(), "All", "l")

legend2.SetBorderSize(0)
legend2.SetFillStyle(0)
legend2.Draw("SAME")

c2.SetLogy()
c2.SetLogx()
c2.SetTitle('Spectra by trigger type (1 GeV-1 TeV) (Unscaled)')
c2.Update()
c2.SaveAs(figPath_ + "scaling/" + "Spectrum_triggers_1_1e5.pdf")
c2.Draw()

Info in <TCanvas::Print>: pdf file /home/users/l/lebrun/figures/scaling/Spectrum_triggers_1_1e5.pdf has been created


## Plot Scaled

### Plot 1 - 100 GeV (Scaled)

In [10]:

n_bins = 30
min_val = 1
max_val = 100

log_bins = create_log_bins(min_val, max_val, n_bins)
log_bins_root = ROOT.std.vector('double')(log_bins)

h_all3 = rdf.Histo1D(("hAll", "Spectra by trigger type (1-100 GeV) (Scaled); Energy [GeV]; Counts", n_bins, log_bins_root.data()), 'E_true', 'weight')
h_none3 = rdf_none.Histo1D(("hNone", "Spectra by trigger type (1-100 GeV) (Scaled); Energy [GeV]; Counts",n_bins, log_bins_root.data()), 'E_true', 'weight')
h_LET3 = rdf_LET.Histo1D(("hLET", "; Energy [GeV]; Counts", n_bins, log_bins_root.data()), 'E_true', 'weight')
h_LETnonHET3 = rdf_LETnonHET.Histo1D(("hLETnonHET", "; Energy [GeV]; Counts", n_bins, log_bins_root.data()), 'E_true', 'weight')
h_HETnonLET3 = rdf_HETnonLET.Histo1D(("hHETnonLET", "; Energy [GeV]; Counts", n_bins, log_bins_root.data()), 'E_true', 'weight')
h_HET3 = rdf_HET.Histo1D(("hHET", "; Energy [GeV]; Counts", n_bins, log_bins_root.data()), 'E_true', 'weight')

In [ ]:

# Style
h_none3.SetLineColor(ROOT.kBlack)
h_LET3.SetLineColor(ROOT.kBlue)
h_LETnonHET3.SetLineColor(ROOT.kCyan+1)
h_HETnonLET3.SetLineColor(ROOT.kRed)
h_HET3.SetLineColor(ROOT.kMagenta)
h_all3.SetLineColor(ROOT.kGreen+2)

h_none3.SetMarkerStyle(1)
h_LET3.SetMarkerStyle(1)
h_LETnonHET3.SetMarkerStyle(1)
h_HETnonLET3.SetMarkerStyle(1)
h_HET3.SetMarkerStyle(1)
h_all3.SetMarkerStyle(1)


h_none3.GetXaxis().SetLabelOffset(0.005)
h_none3.GetXaxis().SetTitleOffset(1.3)
h_LET3.GetXaxis().SetLabelOffset(0.005)
h_LET3.GetXaxis().SetTitleOffset(1.3)
h_LETnonHET3.GetXaxis().SetLabelOffset(0.005)
h_LETnonHET3.GetXaxis().SetTitleOffset(1.3)
h_HETnonLET3.GetXaxis().SetLabelOffset(0.005)
h_HETnonLET3.GetXaxis().SetTitleOffset(1.3)
h_HET3.GetXaxis().SetLabelOffset(0.005)
h_HET3.GetXaxis().SetTitleOffset(1.3)
h_all3.GetXaxis().SetLabelOffset(0.005)
h_all3.GetXaxis().SetTitleOffset(1.3)

# Draw
ROOT.gStyle.SetOptStat(0)
c3 = ROOT.TCanvas("1to100GeVWeighted", "1 to 100 GeV Spectra (Weighted)", 800, 600)

h_none3.SetStats(0)

h_none3.Draw("HIST")
h_LET3.Draw("HIST SAME")
h_LETnonHET3.Draw("HIST SAME")
h_HETnonLET3.Draw("HIST SAME")
h_HET3.Draw("HIST SAME")
h_all3.Draw("HIST SAME")


legend3 = ROOT.TLegend(0.6, 0.6, 0.89, 0.89)  # x1, y1, x2, y2

legend3.AddEntry(h_none.GetPtr(), "None", "l")
legend3.AddEntry(h_LET.GetPtr(), "LET", "l")
legend3.AddEntry(h_LETnonHET.GetPtr(), "LET only", "l")
legend3.AddEntry(h_HETnonLET.GetPtr(), "HET only", "l")
legend3.AddEntry(h_HET.GetPtr(), "HET", "l")
legend3.AddEntry(h_all.GetPtr(), "All", "l")

legend3.SetBorderSize(0)
legend3.SetFillStyle(0)
legend3.Draw("SAME")

c3.SetLogy()
c3.SetLogx()
c3.SetTitle('Spectra by trigger type (1-100 GeV) (Scaled)')
c3.Update()
c3.SaveAs(figPath_ + "scaling/" + "Spectrum_triggers_1_100_scaled.pdf")
c3.Draw()

Info in <TCanvas::Print>: pdf file /home/users/l/lebrun/figures/scaling/Spectrum_triggers_1_100_scaled.pdf has been created


### Plot 1 GeV - 1 TeV (Scaled)

In [ ]:
n_bins = 50
min_val = 1
max_val = 10000

log_bins = create_log_bins(min_val, max_val, n_bins)
log_bins_root = ROOT.std.vector('double')(log_bins)

h_none4 = rdf_none.Histo1D(("hNone", "Spectra by trigger type (1 GeV-10 TeV) (Scaled); Energy [GeV]; Counts",n_bins, log_bins_root.data()), 'E_true', 'weight')
h_LET4 = rdf_LET.Histo1D(("hLET", "; Energy [GeV]; Counts", n_bins, log_bins_root.data()), 'E_true', 'weight')
h_LETnonHET4 = rdf_LETnonHET.Histo1D(("hLETnonHET", "; Energy [GeV]; Counts", n_bins, log_bins_root.data()), 'E_true', 'weight')
h_HETnonLET4 = rdf_HETnonLET.Histo1D(("hHETnonLET", "; Energy [GeV]; Counts", n_bins, log_bins_root.data()), 'E_true', 'weight')
h_HET4 = rdf_HET.Histo1D(("hHET", "; Energy [GeV]; Counts", n_bins, log_bins_root.data()), 'E_true', 'weight')
# Create a histogram for all events
h_all4 = rdf.Histo1D(("hAll", "; Energy [GeV]; Counts", n_bins, log_bins_root.data()), 'E_true', 'weight')


# Style
h_none4.SetLineColor(ROOT.kBlack)
h_LET4.SetLineColor(ROOT.kBlue)
h_LETnonHET4.SetLineColor(ROOT.kCyan+1)
h_HETnonLET4.SetLineColor(ROOT.kRed)
h_HET4.SetLineColor(ROOT.kMagenta)
h_all4.SetLineColor(ROOT.kGreen+2)

h_none4.SetMarkerStyle(1)
h_LET4.SetMarkerStyle(1)
h_LETnonHET4.SetMarkerStyle(1)
h_HETnonLET4.SetMarkerStyle(1)
h_HET4.SetMarkerStyle(1)
h_all4.SetMarkerStyle(1)

h_none4.GetXaxis().SetLabelOffset(0.005)
h_none4.GetXaxis().SetTitleOffset(1.3)
h_LET4.GetXaxis().SetLabelOffset(0.005)
h_LET4.GetXaxis().SetTitleOffset(1.3)
h_LETnonHET4.GetXaxis().SetLabelOffset(0.005)
h_LETnonHET4.GetXaxis().SetTitleOffset(1.3)
h_HETnonLET4.GetXaxis().SetLabelOffset(0.005)
h_HETnonLET4.GetXaxis().SetTitleOffset(1.3)
h_HET4.GetXaxis().SetLabelOffset(0.005)
h_HET4.GetXaxis().SetTitleOffset(1.3)
h_all4.GetXaxis().SetLabelOffset(0.005)
h_all4.GetXaxis().SetTitleOffset(1.3)

# Draw
ROOT.gStyle.SetOptStat(0)
c4 = ROOT.TCanvas("1GeVto10TeVWeighted", "1 GeV to 10 TeV Spectra (Weighted)", 800, 600)

h_none4.SetStats(0)

h_none4.Draw("HIST")
h_LET4.Draw("HIST SAME")
h_LETnonHET4.Draw("HIST SAME")
h_HETnonLET4.Draw("HIST SAME")
h_HET4.Draw("HIST SAME")
h_all4.Draw("HIST SAME")


legend4 = ROOT.TLegend(0.71, 0.6, 0.94, 0.9)  # x1, y1, x2, y2

legend4.AddEntry(h_none.GetPtr(), "None", "l")
legend4.AddEntry(h_LET.GetPtr(), "LET", "l")
legend4.AddEntry(h_LETnonHET.GetPtr(), "LET only", "l")
legend4.AddEntry(h_HETnonLET.GetPtr(), "HET only", "l")
legend4.AddEntry(h_HET.GetPtr(), "HET", "l")
legend4.AddEntry(h_all.GetPtr(), "All", "l")

legend4.SetBorderSize(0)
legend4.SetFillStyle(0)
legend4.Draw("SAME")

c4.SetLogy()
c4.SetLogx()
c4.Update()
c4.SetTitle('Spectra by trigger type (1 GeV-1 TeV) (Scaled)')
c4.SaveAs(figPath_ + "scaling/" + "Spectrum_triggers_1_1e5_scaled.pdf")
c4.Draw()

Info in <TCanvas::Print>: pdf file /home/users/l/lebrun/figures/scaling/Spectrum_triggers_1_1e5_scaled.pdf has been created
